# 📧 Spam-Klassifikation mit TF-IDF, Naive Bayes & Embeddings

**Lernziel:** Schritt-für-Schritt verstehen, wie Textklassifikation funktioniert — von der einfachen Worthäufigkeit (TF-IDF) über klassische ML-Modelle (Naive Bayes) bis zu neuronalen Netzen mit Embeddings.

**Dataset:** [SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) (UCI Machine Learning Repository) — 5.574 echte SMS-Nachrichten, gelabelt als *Spam* oder *Ham* (kein Spam).

**Inhaltsübersicht:**
1. 📦 Setup & Daten laden
2. 📊 Explorative Datenanalyse (EDA)
3. 🔬 TF-IDF: Feature Engineering
4. 🤖 Naive Bayes: Baseline-Modell
5. 📈 Logistic Regression: Vergleich
6. 🧠 Embeddings & Neuronales Netz
7. 📝 Zusammenfassung & nächste Schritte

## 1. 📦 Setup & Daten laden

Wir importieren die benötigten Bibliotheken und laden das SMS Spam Collection Dataset über die Hilfsfunktion `load_spam_data()` aus `spam_classifier.py`.

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Matplotlib für Notebook optimieren
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

# Pfad zum Repo-Root hinzufügen, damit spam_classifier importierbar ist
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, '.')

from spam_classifier import load_spam_data, create_features, evaluate_model

print('✅ Alle Imports erfolgreich!')

In [ ]:
# ── Daten laden ──────────────────────────────────────────────
df = load_spam_data()
print(f'✅ {len(df):,} Nachrichten geladen!')
print(f'   Spam:  {df["label"].sum():,} ({df["label"].mean():.1%})')
print(f'   Ham:   {(df["label"] == 0).sum():,} ({(1 - df["label"].mean()):.1%})')
df.head(10)

## 2. 📊 Explorative Datenanalyse (EDA)

Bevor wir Modelle trainieren, schauen wir uns die Daten genau an: Wie lang sind Spam- vs. Ham-Nachrichten? Welche Wörter kommen häufig vor? Gibt es auffällige Muster?

In [ ]:
# ── Nachrichtenlänge: Spam vs. Ham ───────────────────────────
df['length'] = df['message'].str.len()
df['num_words'] = df['message'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Länge in Zeichen
axes[0].hist(df[df['label'] == 0]['length'], bins=50, alpha=0.6, label='Ham', color='green')
axes[0].hist(df[df['label'] == 1]['length'], bins=50, alpha=0.6, label='Spam', color='red')
axes[0].set_xlabel('Nachrichtenlänge (Zeichen)')
axes[0].set_ylabel('Anzahl')
axes[0].set_title('Verteilung der Nachrichtenlänge', fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0, 300)

# Anzahl Wörter
axes[1].hist(df[df['label'] == 0]['num_words'], bins=50, alpha=0.6, label='Ham', color='green')
axes[1].hist(df[df['label'] == 1]['num_words'], bins=50, alpha=0.6, label='Spam', color='red')
axes[1].set_xlabel('Anzahl Wörter')
axes[1].set_ylabel('Anzahl')
axes[1].set_title('Verteilung der Wortanzahl', fontweight='bold')
axes[1].legend()
axes[1].set_xlim(0, 60)

plt.tight_layout()
plt.show()

print(f'📏 Ham:  Ø {df[df["label"]==0]["length"].mean():.0f} Zeichen, {df[df["label"]==0]["num_words"].mean():.0f} Wörter')
print(f'📏 Spam: Ø {df[df["label"]==1]["length"].mean():.0f} Zeichen, {df[df["label"]==1]["num_words"].mean():.0f} Wörter')

In [ ]:
# ── Beispiel-Nachrichten ─────────────────────────────────────
print('🔴 SPAM-Beispiele:\n' + '─' * 60)
for i, msg in enumerate(df[df['label'] == 1]['message'].sample(5, random_state=42).tolist(), 1):
    print(f'{i}. {msg[:120]}')

print('\n🟢 HAM-Beispiele:\n' + '─' * 60)
for i, msg in enumerate(df[df['label'] == 0]['message'].sample(5, random_state=42).tolist(), 1):
    print(f'{i}. {msg[:120]}')

In [ ]:
# ── Häufigste Wörter in Spam vs. Ham ────────────────────────
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

def top_words(texts, n=15):
    vec = CountVectorizer(stop_words='english', max_features=1000)
    X = vec.fit_transform(texts)
    counts = X.sum(axis=0).A1
    words = vec.get_feature_names_out()
    top_idx = np.argsort(counts)[-n:][::-1]
    return [(words[i], counts[i]) for i in top_idx]

spam_words = top_words(df[df['label'] == 1]['message'])
ham_words = top_words(df[df['label'] == 0]['message'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh([w for w, _ in spam_words], [c for _, c in spam_words], color='red', alpha=0.7)
axes[0].set_title('🔴 Häufigste Wörter in Spam', fontweight='bold')
axes[0].invert_yaxis()

axes[1].barh([w for w, _ in ham_words], [c for _, c in ham_words], color='green', alpha=0.7)
axes[1].set_title('🟢 Häufigste Wörter in Ham', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 3. 🔬 TF-IDF: Feature Engineering

### Was ist TF-IDF?

**TF-IDF** (Term Frequency — Inverse Document Frequency) ist eine Methode, um Text in numerische Vektoren umzuwandeln. Sie gewichtet Wörter danach, wie wichtig sie für ein bestimmtes Dokument sind:

- **TF (Term Frequency):** Wie oft kommt ein Wort in *dieser* Nachricht vor?
  $$\text{TF}(t, d) = \frac{\text{Anzahl von } t \text{ in } d}{\text{Gesamtzahl Wörter in } d}$$

- **IDF (Inverse Document Frequency):** In wie vielen Nachrichten kommt das Wort *insgesamt* vor? Seltene Wörter bekommen höheres Gewicht!
  $$\text{IDF}(t) = \log\frac{N}{|\{d \in D : t \in d\}|}$$

- **TF-IDF:** Produkt aus beiden:
  $$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

**Intuition:** Ein Wort wie "the" kommt in fast jeder Nachricht vor → niedriges IDF → niedriges TF-IDF. Ein Wort wie "prize" kommt nur in wenigen (Spam-)Nachrichten vor → hohes IDF → hohes TF-IDF.

In [ ]:
# ── TF-IDF an einer Beispiel-Nachricht demonstrieren ─────────
from sklearn.feature_extraction.text import TfidfVectorizer

sample_msg = df['message'].iloc[0]
print(f'📝 Beispiel-Nachricht: "{sample_msg}"\n')

# TF-IDF Vectorizer (nur zur Demo, mit wenigen Features)
demo_vectorizer = TfidfVectorizer(max_features=20, stop_words='english')
demo_tfidf = demo_vectorizer.fit_transform([sample_msg])
demo_words = demo_vectorizer.get_feature_names_out()
demo_scores = demo_tfidf.toarray()[0]

# Nur Wörter mit Score > 0 anzeigen
nonzero = demo_scores > 0
if np.any(nonzero):
    indices = np.argsort(demo_scores[nonzero])[::-1]
    print('TF-IDF Scores (nur Wörter mit Score > 0):')
    for idx in indices:
        print(f'  {demo_words[nonzero][idx]:20s} → {demo_scores[nonzero][idx]:.4f}')
else:
    print('Keine Wörter mit TF-IDF > 0 (nur Stopwords?)')

In [ ]:
# ── Vollständige Features extrahieren ────────────────────────
# create_features() aus spam_classifier.py:
# - TF-IDF mit 5000 Features (Unigrams + Bigrams)
# - 7 Metafeatures (Länge, Großbuchstaben, URLs, etc.)

X, vectorizer = create_features(df)
y = df['label'].values

print(f'✅ Feature-Matrix: {X.shape[0]:,} Nachrichten × {X.shape[1]:,} Features')
print(f'   Davon {len(vectorizer.get_feature_names_out()):,} TF-IDF-Features + 7 Metafeatures')
print(f'\n📋 Metafeatures:')
print(f'   - length:           Nachrichtenlänge in Zeichen')
print(f'   - num_words:        Anzahl Wörter')
print(f'   - num_caps:         Anzahl Großbuchstaben')
print(f'   - num_digits:       Anzahl Ziffern')
print(f'   - num_exclamations: Anzahl Ausrufezeichen (!)')
print(f'   - has_url:          Enthält URL? (0/1)')
print(f'   - has_phone:        Enthält Telefonnummer? (0/1)')

## 4. 🤖 Naive Bayes: Baseline-Modell

### Wie funktioniert Naive Bayes?

**Multinomial Naive Bayes** ist ein probabilistischer Klassifikator, der auf dem Bayes-Theorem basiert:

$$P(\text{Spam} \mid \text{Wörter}) = \frac{P(\text{Wörter} \mid \text{Spam}) \cdot P(\text{Spam})}{P(\text{Wörter})}$$

Die "naive" Annahme: Alle Wörter sind **unabhängig** voneinander (was in der Realität nicht stimmt, aber trotzdem erstaunlich gut funktioniert).

**Vorteile:**
- Extrem schnell zu trainieren
- Gut geeignet für Textklassifikation
- Interpretierbar (Wahrscheinlichkeiten pro Wort)

**Nachteile:**
- Die Unabhängigkeitsannahme ist eine Vereinfachung
- Kann die Reihenfolge der Wörter nicht berücksichtigen

In [ ]:
# ── Train/Test-Split ─────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'📊 Train: {X_train.shape[0]:,} Nachrichten')
print(f'📊 Test:  {X_test.shape[0]:,} Nachrichten')
print(f'📊 Spam-Anteil im Test-Set: {y_test.mean():.1%}')

In [ ]:
# ── Naive Bayes trainieren ───────────────────────────────────
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

nb = MultinomialNB(alpha=0.1)  # alpha = Laplace-Smoothing
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print('✅ Naive Bayes trainiert!')
print('\n' + classification_report(y_test, y_pred_nb, target_names=['Ham', 'Spam']))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_nb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Ham', 'Spam'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Naive Bayes — Confusion Matrix', fontweight='bold')
plt.show()

# Fehleranalyse
fp = (y_test == 0) & (y_pred_nb == 1)  # False Positives: Ham → Spam
fn = (y_test == 1) & (y_pred_nb == 0)  # False Negatives: Spam → Ham
print(f'🔍 False Positives (Ham als Spam): {fp.sum()}')
print(f'🔍 False Negatives (Spam als Ham): {fn.sum()}')

## 5. 📈 Logistic Regression: Vergleich

**Logistic Regression** ist ein lineares Modell, das die Wahrscheinlichkeit für Spam als gewichtete Summe der Features berechnet:

$$P(\text{Spam}) = \sigma(w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n)$$

wobei $\sigma$ die **Sigmoid-Funktion** ist, die Werte auf [0, 1] abbildet:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Vorteil gegenüber Naive Bayes:** Die Gewichte $w_i$ sagen uns direkt, wie stark ein Wort Spam-Indikator (positiv) oder Ham-Indikator (negativ) ist.

In [ ]:
# ── Logistic Regression trainieren ───────────────────────────
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print('✅ Logistic Regression trainiert!')
print('\n' + classification_report(y_test, y_pred_lr, target_names=['Ham', 'Spam']))

In [ ]:
# ── Modellvergleich: Naive Bayes vs. Logistic Regression ─────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

models = {
    'Naive Bayes': y_pred_nb,
    'Logistic Regression': y_pred_lr,
}

comparison = []
for name, y_pred in models.items():
    comparison.append({
        'Modell': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
    })

comp_df = pd.DataFrame(comparison).set_index('Modell')
comp_df.style.background_gradient(cmap='Blues', axis=0).format('{:.3f}')

In [ ]:
# ── Visualisierung: Modellvergleich ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.35

for i, (name, row) in enumerate(comp_df.iterrows()):
    values = [row[m] for m in metrics]
    bars = ax.bar(x + i * width, values, width, label=name, alpha=0.8)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Score')
ax.set_title('Modellvergleich: Naive Bayes vs. Logistic Regression', fontweight='bold')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0.85, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Wichtigste Spam- & Ham-Wörter (Logistic Regression) ──────
coef = lr.coef_[0]
feature_names = vectorizer.get_feature_names_out()
n_tfidf = len(feature_names)

# Top-15 Spam-Wörter (höchste positive Koeffizienten)
top_spam_idx = np.argsort(coef[:n_tfidf])[-15:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

spam_words = [feature_names[i] for i in top_spam_idx]
spam_scores = [coef[i] for i in top_spam_idx]
axes[0].barh(range(len(spam_words)), spam_scores, color='red', alpha=0.7)
axes[0].set_yticks(range(len(spam_words)))
axes[0].set_yticklabels(spam_words)
axes[0].set_title('🔴 Top-15 Spam-Wörter', fontweight='bold')
axes[0].set_xlabel('Koeffizient (positiv = Spam-Indikator)')
axes[0].invert_yaxis()

# Top-15 Ham-Wörter (niedrigste/negativste Koeffizienten)
top_ham_idx = np.argsort(coef[:n_tfidf])[:15]
ham_words = [feature_names[i] for i in top_ham_idx]
ham_scores = [coef[i] for i in top_ham_idx]
axes[1].barh(range(len(ham_words)), ham_scores, color='green', alpha=0.7)
axes[1].set_yticks(range(len(ham_words)))
axes[1].set_yticklabels(ham_words)
axes[1].set_title('🟢 Top-15 Ham-Wörter', fontweight='bold')
axes[1].set_xlabel('Koeffizient (negativ = Ham-Indikator)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('💡 Interpretation:')
print('  • Positive Koeffizienten → Wort erhöht Spam-Wahrscheinlichkeit')
print('  • Negative Koeffizienten → Wort spricht für Ham')
print('  • Je größer der Betrag, desto stärker der Einfluss')

## 6. 🧠 Embeddings & Neuronales Netz

### Von TF-IDF zu Embeddings

TF-IDF behandelt Wörter als isolierte Einheiten ("Bag of Words"). **Word Embeddings** gehen einen Schritt weiter: Sie repräsentieren Wörter als dichte Vektoren in einem kontinuierlichen Raum, in dem ähnliche Wörter nah beieinander liegen.

**Beispiel:** Die Wörter "Gewinn", "Preis" und "Geschenk" hätten ähnliche Embedding-Vektoren, weil sie in ähnlichen Kontexten vorkommen.

### Unser Ansatz

Wir bauen ein einfaches neuronales Netz mit:
1. **Embedding-Layer:** Wandelt jedes Wort in einen 100-dimensionalen Vektor um
2. **GlobalAveragePooling1D:** Mittelt die Embeddings aller Wörter → ein Vektor pro Nachricht
3. **Dense-Layer:** Klassifikation (Spam/Ham)

Das ist ein **"Continuous Bag of Words"-Ansatz** — einfach, aber effektiv für die Textklassifikation.

In [ ]:
# ── TensorFlow / Keras importieren ───────────────────────────
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    print(f'✅ TensorFlow {tf.__version__} geladen!')
except ImportError:
    print('⚠️ TensorFlow nicht installiert.')
    print('  Installiere mit: pip install tensorflow')
    print('  Überspringe Embedding-Modell...')

In [ ]:
# ── Text vorbereiten: Tokenisierung & Padding ────────────────
# Nur ausführen, wenn TensorFlow verfügbar ist
if 'tf' in dir():
    # Parameter
    MAX_WORDS = 5000      # Vokabulargröße
    MAX_LEN = 100         # Maximale Nachrichtenlänge (in Tokens)
    EMBEDDING_DIM = 100   # Embedding-Dimension

    # Tokenizer: Text → Sequenz von Integer-IDs
    tokenizer = Tokenizer(num_words=MAX_WORDS)
    tokenizer.fit_on_texts(df['message'])
    sequences = tokenizer.texts_to_sequences(df['message'])

    # Padding: Alle Sequenzen auf gleiche Länge bringen
    X_seq = pad_sequences(sequences, maxlen=MAX_LEN)

    print(f'✅ Tokenisierung abgeschlossen!')
    print(f'   Vokabulargröße: {len(tokenizer.word_index):,} Wörter')
    print(f'   Sequenz-Shape:  {X_seq.shape}')
    print(f'   Beispiel-Sequenz (erste 20 Tokens): {X_seq[0][:20]}')

In [ ]:
# ── Neuronales Netz bauen ────────────────────────────────────
if 'tf' in dir():
    # Train/Test-Split für Sequenzen
    X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
        X_seq, y, test_size=0.2, random_state=42, stratify=y
    )

    # Modell-Architektur
    model = keras.Sequential([
        # Name des Input-Layers für Kompatibilität
        layers.Input(shape=(MAX_LEN,), name='input'),
        # Embedding-Layer: Wort-ID → 100-dimensionaler Vektor
        layers.Embedding(MAX_WORDS, EMBEDDING_DIM, name='embedding'),
        # Global Average Pooling: Mittelwert über alle Wort-Embeddings
        layers.GlobalAveragePooling1D(name='global_avg_pooling'),
        # Hidden Layer
        layers.Dense(64, activation='relu', name='dense_64'),
        layers.Dropout(0.3, name='dropout'),
        # Output Layer: 1 Neuron mit Sigmoid → Spam-Wahrscheinlichkeit
        layers.Dense(1, activation='sigmoid', name='output'),
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy'],
    )

    model.summary()

In [ ]:
# ── Modell trainieren ────────────────────────────────────────
if 'tf' in dir():
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True
    )

    history = model.fit(
        X_train_seq, y_train_seq,
        validation_split=0.1,
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=1,
    )

    print(f'\n✅ Training abgeschlossen! Beste val_accuracy: {max(history.history["val_accuracy"]):.3f}')

In [ ]:
# ── Trainingsverlauf visualisieren ───────────────────────────
if 'tf' in dir() and 'history' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['accuracy'], label='Training', marker='o')
    axes[0].plot(history.history['val_accuracy'], label='Validation', marker='o')
    axes[0].set_title('Accuracy über Epochen', fontweight='bold')
    axes[0].set_xlabel('Epoche')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], label='Training', marker='o')
    axes[1].plot(history.history['val_loss'], label='Validation', marker='o')
    axes[1].set_title('Loss über Epochen', fontweight='bold')
    axes[1].set_xlabel('Epoche')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Embedding-Modell evaluieren ──────────────────────────────
if 'tf' in dir() and 'model' in dir():
    y_pred_nn_prob = model.predict(X_test_seq, verbose=0)
    y_pred_nn = (y_pred_nn_prob > 0.5).astype(int).flatten()

    print('📊 Embedding-Netzwerk — Ergebnisse:')
    print(classification_report(y_test_seq, y_pred_nn, target_names=['Ham', 'Spam']))

    # Vergleich aller drei Modelle
    all_results = pd.DataFrame([
        {'Modell': 'Naive Bayes', 'Accuracy': accuracy_score(y_test, y_pred_nb),
         'Precision': precision_score(y_test, y_pred_nb), 'Recall': recall_score(y_test, y_pred_nb),
         'F1-Score': f1_score(y_test, y_pred_nb)},
        {'Modell': 'Logistic Regression', 'Accuracy': accuracy_score(y_test, y_pred_lr),
         'Precision': precision_score(y_test, y_pred_lr), 'Recall': recall_score(y_test, y_pred_lr),
         'F1-Score': f1_score(y_test, y_pred_lr)},
        {'Modell': 'Embedding-Netzwerk', 'Accuracy': accuracy_score(y_test_seq, y_pred_nn),
         'Precision': precision_score(y_test_seq, y_pred_nn), 'Recall': recall_score(y_test_seq, y_pred_nn),
         'F1-Score': f1_score(y_test_seq, y_pred_nn)},
    ]).set_index('Modell')

    all_results.style.background_gradient(cmap='Blues', axis=0).format('{:.3f}')

In [ ]:
# ── Live-Vorhersage: Eigene Texte testen ─────────────────────
def predict_spam(text, model_choice='lr'):
    """
    Sagt vorher, ob ein Text Spam oder Ham ist.

    Args:
        text: Der zu klassifizierende Text
        model_choice: 'nb' (Naive Bayes), 'lr' (Logistic Regression), 'nn' (Embedding-Netzwerk)
    """
    from scipy.sparse import csr_matrix, hstack

    if model_choice == 'nn' and 'model' in dir():
        seq = tokenizer.texts_to_sequences([text])
        seq_padded = pad_sequences(seq, maxlen=MAX_LEN)
        proba = model.predict(seq_padded, verbose=0)[0][0]
        pred = 1 if proba > 0.5 else 0
    else:
        # TF-IDF + Metafeatures
        text_tfidf = vectorizer.transform([text])
        meta = np.array([[
            len(text), len(text.split()),
            sum(1 for c in text if c.isupper()),
            sum(1 for c in text if c.isdigit()),
            text.count('!'),
            1 if any(w in text.lower() for w in ['http', 'www.']) else 0,
            1 if any(c.isdigit() for c in text) and len([c for c in text if c.isdigit()]) >= 3 else 0,
        ]])
        X_input = hstack([text_tfidf, csr_matrix(meta)])

        if model_choice == 'nb':
            proba = nb.predict_proba(X_input)[0][1]
            pred = nb.predict(X_input)[0]
        else:
            proba = lr.predict_proba(X_input)[0][1]
            pred = lr.predict(X_input)[0]

    label = '🔴 SPAM' if pred == 1 else '🟢 HAM'
    print(f'{label} (Wahrscheinlichkeit: {proba:.1%})')
    return pred, proba

# Teste ein paar Beispiele
print('─' * 60)
print('Test 1 — Klassisches Spam:')
predict_spam('Congratulations! You have won a FREE iPhone. Click here to claim your prize now!')

print('\nTest 2 — Normale Nachricht:')
predict_spam('Hey, are we still meeting for lunch tomorrow at 12?')

print('\nTest 3 — Dringende Spam-Nachricht:')
predict_spam('URGENT! Your account has been compromised. Call 0900-123-456 immediately to verify.')

print('\nTest 4 — Harmlose Frage:')
predict_spam('Can you pick up some milk on your way home? Thanks!')

## 7. 📝 Zusammenfassung & nächste Schritte

### Was wir gelernt haben

| Konzept | Beschreibung |
|---------|-------------|
| **TF-IDF** | Wandelt Text in numerische Vektoren um; gewichtet seltene Wörter höher |
| **Naive Bayes** | Probabilistischer Klassifikator; schnell & einfach, gute Baseline |
| **Logistic Regression** | Lineares Modell; Gewichte direkt interpretierbar als Wort-Wichtigkeit |
| **Embeddings** | Dichte Vektorrepräsentation; erfasst semantische Ähnlichkeit |
| **Neuronales Netz** | Löst nicht-lineare Klassifikation; braucht mehr Daten & Rechenzeit |

### Nächste Schritte

- **Fortgeschrittene Embeddings:** Vortrainierte Modelle wie BERT oder FastText verwenden
- **LSTM / GRU:** Sequenzielle Modelle, die die Wortreihenfolge berücksichtigen
- **Hyperparameter-Tuning:** Grid Search für alpha (Naive Bayes), C (Logistic Regression)
- **Cross-Validation:** Robustere Evaluation mit k-Fold CV
- **Weitere Datasets:** Eigene E-Mail-Daten oder das Enron-Spam-Dataset

---
*Notebook erstellt mit den Modulen aus `spam_classifier.py` · SMS Spam Collection Dataset (UCI)*